In [ ]:
# Dependencies are installed by the Hendrix Slurm script.


In [ ]:
import os
import re
import time
import random
import shutil
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, concatenate_datasets, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

os.environ["WANDB_DISABLED"] = "true"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

In [ ]:
ds = load_dataset("contemmcm/sentiment140")

ds

In [ ]:
data = ds["complete"]

data[0]

In [ ]:
SUBSAMPLE_SIZE = 50_000

In [ ]:
negative = data.filter(lambda x: x["label"] == 0)
positive = data.filter(lambda x: x["label"] == 1)

n_per_class = SUBSAMPLE_SIZE // 2

negative_sample = negative.shuffle(seed=SEED).select(range(n_per_class))
positive_sample = positive.shuffle(seed=SEED).select(range(n_per_class))

balanced_data = concatenate_datasets([negative_sample, positive_sample]).shuffle(seed=SEED)

print("Balanced sample size:", len(balanced_data))

In [ ]:
df = pd.DataFrame({
    "text": balanced_data["text"],
    "label": [0 if y == 0 else 1 for y in balanced_data["label"]]
})

df.head()

In [ ]:
df["label"].value_counts()

In [ ]:
def clean_tweet(text):
    text = str(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Replace @usernames with @user
    text = re.sub(r"@\w+", "@user", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["clean_text"] = df["text"].apply(clean_tweet)

df[["text", "clean_text", "label"]].head(10)

In [ ]:
X = df["clean_text"].tolist()
y = df["label"].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

In [ ]:
train_df = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

test_df = pd.DataFrame({
    "text": X_test,
    "label": y_test
})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset, test_dataset

In [ ]:
train_df = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

test_df = pd.DataFrame({
    "text": X_test,
    "label": y_test
})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset, test_dataset

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
MAX_LENGTH = 64

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_tokenized = train_dataset.map(tokenize_batch, batched=True)
test_tokenized = test_dataset.map(tokenize_batch, batched=True)

In [ ]:
print("Train columns:", train_tokenized.column_names)
print("Test columns:", test_tokenized.column_names)



In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "f1": f1
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./distilbert_sentiment140_results_cosine_fgm_eps02",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)

In [ ]:
class EmbeddingPerturbation:
    def __init__(self, model):
        self.model = model
        self.backup = {}

    def attack(self, epsilon=0.2, emb_name="word_embeddings"):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    param.data.add_(epsilon * param.grad / norm)

    def restore(self, emb_name="word_embeddings"):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data.copy_(self.backup[name])
        self.backup = {}


class EmbeddingPerturbationTrainer(Trainer):
    def __init__(self, *args, perturbation_epsilon=0.2, perturbation_emb_name="word_embeddings", **kwargs):
        super().__init__(*args, **kwargs)
        self.perturbation_epsilon = perturbation_epsilon
        self.perturbation_emb_name = perturbation_emb_name
        self.embedding_perturbation = EmbeddingPerturbation(self.model)

    def _compute_loss_for_perturbation(self, model, inputs, num_items_in_batch=None):
        kwargs = {}
        if num_items_in_batch is not None:
            kwargs["num_items_in_batch"] = num_items_in_batch
        return self.compute_loss(model, inputs, **kwargs)

    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        if hasattr(self.optimizer, "train") and callable(self.optimizer.train):
            self.optimizer.train()

        inputs = self._prepare_inputs(inputs)

        with self.compute_loss_context_manager():
            loss = self._compute_loss_for_perturbation(model, inputs, num_items_in_batch)
        if self.args.n_gpu > 1:
            loss = loss.mean()
        self.accelerator.backward(loss / self.args.gradient_accumulation_steps)

        self.embedding_perturbation.attack(
            epsilon=self.perturbation_epsilon,
            emb_name=self.perturbation_emb_name,
        )
        with self.compute_loss_context_manager():
            adversarial_loss = self._compute_loss_for_perturbation(model, inputs, num_items_in_batch)
        if self.args.n_gpu > 1:
            adversarial_loss = adversarial_loss.mean()
        self.accelerator.backward(adversarial_loss / self.args.gradient_accumulation_steps)
        self.embedding_perturbation.restore(emb_name=self.perturbation_emb_name)

        return loss.detach()


In [ ]:
trainer = EmbeddingPerturbationTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    perturbation_epsilon=0.2
)

In [ ]:
start_time = time.time()

trainer.train()

train_time = time.time() - start_time

print(f"Training time: {train_time:.2f} seconds")

In [ ]:
eval_results = trainer.evaluate()

eval_results

In [ ]:
pred_output = trainer.predict(test_tokenized)

logits = pred_output.predictions
y_pred = np.argmax(logits, axis=1)
y_true = pred_output.label_ids

test_accuracy = accuracy_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred)

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test F1: {test_f1:.4f}")

In [ ]:
print(classification_report(
    y_true,
    y_pred,
    target_names=["negative", "positive"]
))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Negative", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Positive"]
)

cm_df

In [ ]:
model.eval()

test_texts = X_test

start_time = time.time()

for text in test_texts:
    clean = clean_tweet(text)

    inputs = tokenizer(
        clean,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        _ = model(**inputs)

total_inference_time = time.time() - start_time
latency_per_example_ms = (total_inference_time / len(test_texts)) * 1000

print(f"Total inference time: {total_inference_time:.4f} seconds")
print(f"Inference latency: {latency_per_example_ms:.4f} ms/example")

In [ ]:
save_dir = "distilbert_sentiment140_model_cosine_fgm_eps02"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("Saved model to:", save_dir)

In [ ]:
def get_dir_size_mb(path):
    total_size = 0

    for dirpath, dirnames, filenames in os.walk(path):
        for filename in filenames:
            file_path = os.path.join(dirpath, filename)
            total_size += os.path.getsize(file_path)

    return total_size / (1024 * 1024)

model_size_mb = get_dir_size_mb(save_dir)

print(f"Model size: {model_size_mb:.2f} MB")

In [ ]:
transformer_results = {
    "Model": "DistilBERT fine-tune",
    "Dataset Size": len(df),
    "Train Size": len(X_train),
    "Test Size": len(X_test),
    "Accuracy": test_accuracy,
    "F1": test_f1,
    "Train Time (s)": train_time,
    "Inference (ms/example)": latency_per_example_ms,
    "Model Size (MB)": model_size_mb,
    "Max Tokens": MAX_LENGTH,
    "Epochs": 2
}

transformer_results_df = pd.DataFrame([transformer_results])

transformer_results_df

In [ ]:
transformer_results_df.to_csv("transformer_model_results.csv", index=False)

print("Saved results to transformer_model_results.csv")